In [51]:
import os
import torch

In [ ]:
with open("names.txt", "r+") as file:
    contents = file.read()

print(contents)

In [42]:
split_line_list = contents.splitlines()


handled = []
for word in split_line_list:
    word_split = list(word)
    word_split.insert(0, '<S>')
    word_split.append('<E>')
    handled.append(word_split)


In [4]:
b = {}
for chs in handled:
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

print(len(b))
print(sorted(b.items(), key=lambda kv: -kv[1])[:10])
        

    

627
[(('n', '<E>'), 6763), (('a', '<E>'), 6640), (('a', 'n'), 5438), (('<S>', 'a'), 4410), (('e', '<E>'), 3983), (('a', 'r'), 3264), (('e', 'l'), 3248), (('r', 'i'), 3033), (('n', 'a'), 2977), (('<S>', 'k'), 2963)]


In [50]:
chars = sorted(set(''.join(split_line_list)))

char_to_index = {}
index_to_char = {}

char_to_index['.'] = 0

for i, ch in enumerate(chars):
    char_to_index[ch] = i + 1

for ch, i in char_to_index.items():
    index_to_char[i] = ch


N = torch.zeros((27, 27), dtype=torch.int32)
for pair, num in b.items():
    (start, end) = pair
    t_start = '.' if start == '<S>' or start == '<E>' else start
    t_end = '.' if end == '<S>' or end == '<E>' else end
    x = char_to_index[t_start]
    y = char_to_index[t_end]

    N[x, y] = num

P = (N + 1) / (N + 1).sum(1, keepdim=True)
g = torch.Generator().manual_seed(2147483647)

def generate_name(gen): 
    prob = []
    final_char = ''
    ix = 0
    while(True):
        p = P[ix]
        index = torch.multinomial(p, num_samples=1, replacement=True, generator=gen)
        ix = index.item()
        if ix == 0:
            break 
        final_char += index_to_char[ix]
        prob.append(p[index])
    return final_char, prob
    
r = generate_name(g)

def culculate_loss(name, P, ch2i):
    pair_num = len(name) + 1
    if pair_num == 0:
        return
    log_sum = 0.0
    cons_name = ['.'] + list(name) + ['.']
    pairs = zip(cons_name, cons_name[1:])
    probs = []
    for s, e in pairs:
        x = ch2i[s]
        y = ch2i[e]
        probs.append(P[x, y])
    
    for prob in probs:
        log_sum += -torch.log(prob).item()
        
    return log_sum, pair_num

names = ['andrejq', 'emma', 'tom']
total_log_sum = 0.0
total_pair_num = 0
for n in names:
    (log_sum, pair_num) = culculate_loss(n, P, char_to_index)
    print(log_sum)
    total_log_sum += log_sum
    total_pair_num += pair_num
    
print(total_log_sum / total_pair_num)



27.867215991020203
12.571641862392426
11.295191049575806
3.043179347234614
